## Human-In-The-Loop Workflows with Microsoft Agent Framework and Foundry Agents

### Installing Utilities and Libraries

In [ ]:
%pip install agent-framework==1.19.0 azure-ai-projects==2.7.0 azure-identity==1.25.3 azure-monitor-opentelemetry==1.8.8

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential

load_dotenv()
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")

credential = AzureCliCredential()

### Creating the Foundry Chat Client

In [ ]:
from agent_framework import Agent
from azure.ai.projects import AIProjectClient
from agent_framework.foundry import FoundryAgent, FoundryChatClient, to_prompt_agent
from azure.identity import AzureCliCredential
from agent_framework import (
    Agent,
    AgentExecutor,
    AgentExecutorResponse,
    Content,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowEvent,
    handler,
    response_handler,
    WorkflowViz
)
from collections.abc import AsyncIterable

client = FoundryChatClient(
    project_endpoint = foundry_project_endpoint,
    model = model_deployment_name,
    credential = credential
)

await client.configure_azure_monitor(
    enable_sensitive_data=True
)

### Define the MCP Server Tool

In [ ]:
microsoft_learn_mcp = client.get_mcp_tool(
    name=mcp_server_name,
    url="https://learn.microsoft.com/api/mcp",
    approval_mode="always_require"
)

### Create the Research Agent

In [ ]:
research_agent = Agent(
    client=client,

    name="MicrosoftResearchAgent",

    instructions="""
    You are a Microsoft technical research specialist.

    Research the user's requested Microsoft technology
    using the Microsoft Learn MCP server.

    You MUST use Microsoft Learn MCP tools to obtain
    authoritative information before answering.

    Gather the most relevant technical information.

    Your research notes should include:

    - important concepts
    - architecture
    - important APIs or SDK features
    - implementation details
    - practical use cases

    Produce detailed research notes.

    Do NOT write the final article.
    Another agent will do that.
    """,

    tools=[
        microsoft_learn_mcp
    ]
)

### Create the Writer Agent

In [ ]:
writer_agent = Agent(
    client=client,

    name="TechnicalWriterAgent",

    instructions="""
    You are a senior Microsoft technical writer.

    You will receive research notes that have already
    been reviewed and approved by a human.

    Convert the research into a polished technical article.

    The article should:

    - have a clear title
    - explain the topic clearly
    - use appropriate technical terminology
    - explain important APIs and concepts
    - include practical examples where appropriate
    - be well structured
    - remain faithful to the supplied research

    Do not perform additional research.

    Return only the final article.
    """
)#

### Define Workflow Review Request

In [ ]:
from dataclasses import dataclass

@dataclass
class ResearchApprovalRequest:

    instructions: str

    research: str

### Define the Human Research Review Executor

In [ ]:
class HumanResearchReview(Executor):

    def __init__(self):

        super().__init__(
            id="human_research_review"
        )


    @handler
    async def request_review(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext
    ) -> None:

        research = (
            response.agent_response.text
        )

        await ctx.request_info(

            request_data=ResearchApprovalRequest(

                instructions=(
                    "Review the research generated by "
                    "the ResearchAgent. Enter APPROVE "
                    "to allow the WriterAgent to continue "
                    "or REJECT to stop the workflow."
                ),

                research=research
            ),

            response_type=str
        )


    @response_handler
    async def process_review(
        self,
        original_request: ResearchApprovalRequest,
        decision: str,
        ctx: WorkflowContext
    ) -> None:

        decision = decision.strip().upper()

        if decision == "APPROVE":

            await ctx.send_message(
                original_request.research
            )

        else:

            await ctx.yield_output(
                "WORKFLOW REJECTED BY HUMAN REVIEWER"
            )

### Create the Agent Executors

In [ ]:
research_executor = AgentExecutor(
    research_agent
)

writer_executor = AgentExecutor(
    writer_agent
)

human_review = HumanResearchReview()

### Build the Workflow

In [ ]:
workflow = (
    WorkflowBuilder(
        start_executor=research_executor
    )

    .add_edge(
        research_executor,
        human_review
    )

    .add_edge(
        human_review,
        writer_executor
    )

    .build()
)

viz = WorkflowViz(workflow)

### Visualize the Workflow

In [ ]:
mermaid_content = viz.to_mermaid()

# printing mermaid content as markdown
from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{mermaid_content}\n```"))

### Define the Research Task

In [ ]:
query = """
Research how Microsoft Agent Framework supports
human-in-the-loop workflows.

Use Microsoft Learn documentation.

Focus specifically on:

- request and response handling
- request_info()
- workflow suspension
- workflow resumption
- tool approval
- function approval requests
- practical enterprise use cases

Prepare detailed research notes that can later
be converted into a technical article.
"""

### Process Workflow Events

In [ ]:
async def process_events(
    stream: AsyncIterable[WorkflowEvent]
):

    pending_responses = {}

    async for event in stream:

        print(
            f"Event: {event.type}"
        )

        # Human Input Required
        if event.type == "request_info":

            request = event.data

            # Case 1:
            # MCP Tool Approval Request
            if (
                isinstance(request, Content)
                and
                request.type
                == "function_approval_request"
            ):

                function_call = (
                    request.function_call
                )

                print(
                    "\n================================"
                )

                print(
                    "MCP TOOL APPROVAL REQUIRED"
                )

                print(
                    "================================\n"
                )

                print(
                    "Tool:",
                    function_call.name
                )

                print(
                    "\nArguments:"
                )

                print(
                    function_call.arguments
                )

                print()

                decision = input(
                    "Approve MCP tool call? (Y/N): "
                )

                approved = (
                    decision.strip().lower()
                    in ["y", "yes"]
                )


                pending_responses[
                    event.request_id
                ] = (
                    request
                    .to_function_approval_response(
                        approved=approved
                    )
                )

            # Case 2:
            # Worklow-level human review
            elif isinstance(
                request,
                ResearchApprovalRequest
            ):

                print(
                    "\n================================"
                )

                print(
                    "WORKFLOW HUMAN REVIEW"
                )

                print(
                    "================================\n"
                )

                print(
                    request.instructions
                )

                print(
                    "\n--- RESEARCH ---\n"
                )

                print(
                    request.research
                )

                print(
                    "\n----------------\n"
                )

                decision = input(
                    "APPROVE / REJECT: "
                )

                pending_responses[
                    event.request_id
                ] = decision

        # Final workflow output
        elif event.type == "output":

            print(
                event.data
            )


    return (
        pending_responses
        if pending_responses
        else None
    )

### Run the Workflow

In [ ]:
async def run_hitl_workflow(
    workflow,
    initial_input
):

    # Start workflow
    stream = workflow.run(
        initial_input,
        stream=True
    )


    pending_responses = (
        await process_events(
            stream
        )
    )

    # Resume while human input is required
    while pending_responses is not None:

        stream = workflow.run(
            stream=True,
            responses=pending_responses
        )

        pending_responses = (
            await process_events(
                stream
            )
        )

In [ ]:
await run_hitl_workflow(
    workflow,
    query
)